# Clase 4 · 07 · Comparación batch y recuperación con checkpoints
Ejecutar después de 005. Completá el punto A, ejecutá B y explicá ambos resultados en tu entrega.
El replay usa una tabla y rutas propias; no cambia el pipeline principal.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")
dbutils.widgets.text("experiment_id", "exp01", "05 - Experimento de replay")
dbutils.widgets.dropdown("validar_entrega", "no", ["no", "si"], "06 - Validar desafío")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
%run ../common/streaming_expectations

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

In [ ]:
assert spark.table("stream_run_audit").where("expected_batch_id = 'stream_005'").count() > 0, "Terminá las cinco etapas."

## A. Reconstrucción batch del histórico válido
Construí una vista temporal `stream_batch_reference` desde `silver_stream_valid`:
sólo purchase, deduplicación por event_id y ventanas de 5 minutos por canal, sin watermark.
Columnas: window_start y window_end TIMESTAMP, payment_channel STRING,
event_count BIGINT, total_amount DECIMAL(18,2), fraud_count BIGINT.

Compará cada ventana/canal con Gold mediante un full join. Explicá qué evento genera la diferencia,
por qué su registro era válido y en qué etapa quedó excluido de streaming.
El control verifica los tipos, ambas fechas de cada ventana y todas las métricas por canal, incluido fraud_count.


In [ ]:
# TODO: construir el DataFrame batch y createOrReplaceTempView("stream_batch_reference").

## B. Reiniciar con el mismo checkpoint y con otro
Ejecutá el experimento provisto. Se leen los mismos cinco archivos en una tabla de replay nueva:
primera ejecución con checkpoint a; segunda con a; tercera con b.
No borres checkpoints ni cambies rutas del pipeline principal.

Para repetir desde cero, elegí un nuevo experiment_id. Si una ejecución falla, reejecutá sólo
la parte correspondiente con las rutas ya asignadas; no cambies de checkpoint para ocultar el error.

In [ ]:
import re
experiment_id = dbutils.widgets.get("experiment_id").strip().lower()
assert re.fullmatch(r"[a-z][a-z0-9_]{1,20}", experiment_id), "Usá 2–21 caracteres: letra inicial, letras, números y _."
replay_table = qualified_table(config, f"bronze_stream_replay_{experiment_id}")
assert not spark.catalog.tableExists(replay_table), "Elegí otro experiment_id para repetir el desafío desde cero."
replay_source = (spark.readStream.format("cloudFiles").option("cloudFiles.format", "text")
                 .option("cloudFiles.useNotifications", "false").schema("value STRING")
                 .load(stream_input_directory(config))
                 .select(F.col("value").alias("raw_json"), F.col("_metadata.file_path").alias("source_file")))

In [ ]:
def run_replay(checkpoint_label):
    query = (replay_source.writeStream.format("delta").outputMode("append")
             .option("checkpointLocation", f"{config.volume_path}/streaming_lab/checkpoints/desafio/{experiment_id}/{checkpoint_label}")
             .trigger(availableNow=True).toTable(replay_table))
    wait_and_record_stream(spark, config, query, "replay", job_run_id)
    return spark.table(replay_table).count()

In [ ]:
first = run_replay("checkpoint_a")

In [ ]:
second = run_replay("checkpoint_a")

In [ ]:
third = run_replay("checkpoint_b")

In [ ]:
assert (first, second, third) == (14 * repetitions, 14 * repetitions, 28 * repetitions)
display(spark.createDataFrame([(1, "a", first), (2, "a", second), (3, "b", third)],
                             ["attempt", "checkpoint", "output_rows"]))

## C. Explicación de recuperación
Escribí un plan breve para un fallo entre Bronze y Gold:
qué tareas repetir, qué checkpoints conservar y cómo verificar que el resultado es consistente.
Justificá por qué un checkpoint nuevo con el mismo destino append puede duplicar datos.
Indicá qué tendría que cambiar si modificamos las claves de deduplicación o el grano de la agregación.

In [ ]:
if dbutils.widgets.get("validar_entrega") != "si":
    print("Validación pendiente: resolvé A y B, luego seleccioná validar_entrega = si.")
else:
    from decimal import Decimal
    assert spark.catalog.tableExists("stream_batch_reference"), "Falta resolver el punto A en esta sesión."
    reference = spark.table("stream_batch_reference")
    def validate_batch_reference(reference, repetitions):
        from datetime import timedelta
        from decimal import Decimal

        expected_types = {
            "window_start": "timestamp", "window_end": "timestamp",
            "payment_channel": "string", "event_count": "bigint",
            "total_amount": "decimal(18,2)", "fraud_count": "bigint",
        }
        assert dict(reference.dtypes) == expected_types, "Revisá las columnas y tipos de la referencia batch."
        expected = expected_gold_rows("stream_005", repetitions)
        late_key = ("2026-03-12 12:00:00", "card")
        count, amount, fraud = expected[late_key]
        expected[late_key] = (count + repetitions, amount + Decimal("999.00") * repetitions, fraud)
        actual = {}
        for row in reference.collect():
            assert row.window_start is not None and row.window_end is not None, "Una ventana no puede ser NULL."
            assert row.window_end == row.window_start + timedelta(minutes=5), "La ventana debe durar cinco minutos."
            key = (row.window_start.strftime("%Y-%m-%d %H:%M:%S"), row.payment_channel)
            assert key not in actual, "Hay más de una fila por ventana/canal."
            actual[key] = (row.event_count, row.total_amount, row.fraud_count)
        assert actual == expected, f"Referencia batch incorrecta por ventana/canal: {actual}"

    validate_batch_reference(reference, repetitions)
    diff = (reference.alias("batch").join(spark.table("gold_stream_windows").alias("stream"),
            ["window_start", "window_end", "payment_channel"], "full")
            .select("window_start", "payment_channel",
                    (F.coalesce(F.col("batch.event_count"), F.lit(0)) - F.coalesce(F.col("stream.event_count"), F.lit(0))).alias("extra_events"),
                    (F.coalesce(F.col("batch.total_amount"), F.lit(0)) - F.coalesce(F.col("stream.total_amount"), F.lit(0))).alias("extra_amount"),
                    (F.coalesce(F.col("batch.fraud_count"), F.lit(0)) - F.coalesce(F.col("stream.fraud_count"), F.lit(0))).alias("extra_frauds")))
    changed = diff.where("extra_events <> 0 OR extra_amount <> 0 OR extra_frauds <> 0").collect()
    assert len(changed) == 1
    assert changed[0].payment_channel == "card"
    assert changed[0].window_start.strftime("%Y-%m-%d %H:%M:%S") == "2026-03-12 12:00:00"
    assert changed[0].extra_events == repetitions and changed[0].extra_amount == Decimal("999.00") * repetitions
    assert changed[0].extra_frauds == 0
    assert spark.table(replay_table).count() == 28 * repetitions, "Falta resolver el punto B."
    assert spark.table("bronze_stream_events").count() == 14 * repetitions
    print("Desafío validado; agregá tu explicación sobre las diferencias de semántica y recuperación.")

## Entrega
Exportá el notebook con sus salidas y el control final aprobado. En el README agregá
la comparación batch/stream, las tres cantidades del replay y el plan de recuperación.
Ver la estructura completa de `resolucion-practica-4/` y las 20 preguntas en README del TP.